# Machine Learning Model (Learns Peer Patterns)

In [2]:
# import neccessary libraries

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
import pandas as pd
import numpy as np
from pathlib import Path


In [3]:
# Load the dataset

# Path.cwd() gets the 'modeling' directory, and .parent moves up to the project root
project_root = Path.cwd().parent

# The division operator (/) is used by pathlib to join paths safely on any OS
gold_path = project_root / "lakehouse" / "gold" / "gold_with_peers.parquet"

# Read the parquet file into a DataFrame
gold = pd.read_parquet(gold_path)

print("Datasets loaded successfully:")

Datasets loaded successfully:


In [8]:
gold.columns
gold.info()

<class 'pandas.DataFrame'>
RangeIndex: 2334830 entries, 0 to 2334829
Data columns (total 34 columns):
 #   Column                 Dtype   
---  ------                 -----   
 0   Outlet_ID              str     
 1   Year                   int16   
 2   Month                  int8    
 3   Distributor_ID         str     
 4   SKU_ID                 str     
 5   Volume_Liters          float32 
 6   Total_Bill_Value       float64 
 7   Outlet_Size            category
 8   Cooler_Count           float32 
 9   Outlet_Type            category
 10  Latitude               float32 
 11  Longitude              float32 
 12  Seasonality_Index      category
 13  Holiday_Count          int64   
 14  poi_bus_stop_1500m     int8    
 15  poi_hospital_1500m     int8    
 16  poi_market_1500m       int8    
 17  poi_school_1500m       int8    
 18  poi_supermarket_1500m  int8    
 19  poi_tourism_1500m      int16   
 20  mean_vol               float64 
 21  max_vol                float64 
 22  min_v

In [ ]:
# Building the model

# ── Training set: unconstrained outlets only ──────────────────────────────
# Their Jan-adjusted max IS their potential (ground truth proxy)
gold['jan_target'] = gold['max_vol'] * gold['Seasonality_Index'].fillna(1.0)

feature_cols = [
    # Historical volume signals
    'mean_vol', 'max_vol', 'p90_vol', 'p10_vol',
    'CV', 'Ceiling_Hitting_Rate', 'Zero_Month_Rate',
    # POI features
    'poi_bus_stop_1500m', 'poi_hospital_1500m',
    'poi_market_1500m', 'poi_school_1500m', 'poi_supermarket_1500m',
    'poi_tourism_1500m',
    # Seasonality
    'Seasonality_Index', 'Holiday_Count',
    # Encoded categoricals
    #'outlet_type_enc', 'province_enc',
]

train = gold[~gold['is_constrained']].copy()
X_train = train[feature_cols].fillna(0)
y_train = train['jan_target']

# Cross-validate to verify the model is learning something real
model = GradientBoostingRegressor(
    n_estimators=400,
    learning_rate=0.04,
    max_depth=4,
    subsample=0.75,
    min_samples_leaf=10,
    random_state=42
)

cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
print(f"CV R² scores: {cv_scores}")
print(f"Mean R²: {cv_scores.mean():.3f}")

model.fit(X_train, y_train)

TypeError: Object with dtype category cannot perform the numpy op multiply